# THUOC - FitLap Notebook (NVIDIA GPU, Updated 2026-04-02)

Notebook nay da duoc viet lai de chay tren may FitLap/Linux co GPU NVIDIA.

## Muc tieu
- Chay train va evaluate dung pipeline THUOC tren moi truong local/venv.
- Khong phu thuoc Google Colab hoac Google Drive.
- Van tai su dung ky thuat du lieu va huan luyen tu Lab6 (khong dung model Lab6).

## Cach dung nhanh
1. Kich hoat venv va mo notebook trong thu muc du an THUOC.
2. Chay lan luot tu Cell 1 den Cell 11.
3. Neu chi muon danh gia checkpoint da co, dat RUN_MODE = compare-only.
4. Neu muon train 1 model, dat RUN_MODE = single va chon SINGLE_MODEL.

In [ ]:
# Cell 1 - Cau hinh FitLap (Linux + NVIDIA GPU)
from pathlib import Path
import os

PROJECT_NAME = "THUOC"
PROJECT_DIR = Path(
    os.environ.get(
        "THUOC_PROJECT_DIR",
        str(Path.home() / "iDragonCloud" / PROJECT_NAME),
    )
).expanduser()

# full | compare-only | single
RUN_MODE = "full"
SINGLE_MODEL = "resnet50"  # dung khi RUN_MODE = single
SINGLE_EPOCHS = 12

# Git settings (tuy chon)
UPDATE_REPO = False
REPO_BRANCH = "main"

# Data settings
PREFERRED_DATA_DIR = "data_aligned"
EXTRA_DATA_CANDIDATES = []  # Them path neu can, vi du: ["/data/thuoc/data_aligned"]

# Runtime settings
DEVICE = "cuda"  # Se tu fallback sang CPU neu CUDA khong co
BATCH_SIZE = 16
NUM_WORKERS = 4
EVAL_BATCH_SIZE = 0  # 0 = auto
MAX_EVAL_SAMPLES = 0  # 0 = full test set
SEED = 42
FAST_TRAIN = False
SKIP_METADATA_ARTIFACTS = False

# Lab6-inspired data and training technologies (khong su dung model Lab6)
LAB6_TECH_MODE = True
AUGMENT_PROFILE = "default"
VECTOR_GRAYSCALE_PROB = 0.0
PREVIEW_RANDOM_TRAIN_SAMPLES = True
RANDOM_SAMPLE_COUNT = 9
BATCH_SIZE_CANDIDATES = [16, 24, 32]

# Data quality gate
ALLOW_EMPTY_VALTEST_CLASSES = True  # Dat False neu bat buoc moi class phai co anh o val/test

if LAB6_TECH_MODE and RUN_MODE != "compare-only":
    AUGMENT_PROFILE = "lab6_stable"
    VECTOR_GRAYSCALE_PROB = 0.15

def print_config_block() -> None:
    print("=" * 72)
    print("THUOC FITLAP CONFIG")
    print("=" * 72)
    print(f"PROJECT_DIR              : {PROJECT_DIR}")
    print(f"RUN_MODE                 : {RUN_MODE}")
    print(f"SINGLE_MODEL             : {SINGLE_MODEL}")
    print(f"SINGLE_EPOCHS            : {SINGLE_EPOCHS}")
    print(f"UPDATE_REPO              : {UPDATE_REPO}")
    print(f"REPO_BRANCH              : {REPO_BRANCH}")
    print(f"DEVICE                   : {DEVICE}")
    print(f"BATCH_SIZE               : {BATCH_SIZE}")
    print(f"NUM_WORKERS              : {NUM_WORKERS}")
    print(f"EVAL_BATCH_SIZE          : {EVAL_BATCH_SIZE}")
    print(f"MAX_EVAL_SAMPLES         : {MAX_EVAL_SAMPLES}")
    print(f"FAST_TRAIN               : {FAST_TRAIN}")
    print(f"SKIP_METADATA_ARTIFACTS  : {SKIP_METADATA_ARTIFACTS}")
    print(f"PREFERRED_DATA_DIR       : {PREFERRED_DATA_DIR}")
    print(f"EXTRA_DATA_CANDIDATES    : {EXTRA_DATA_CANDIDATES}")
    print(f"LAB6_TECH_MODE           : {LAB6_TECH_MODE}")
    print(f"AUGMENT_PROFILE          : {AUGMENT_PROFILE}")
    print(f"VECTOR_GRAYSCALE_PROB    : {VECTOR_GRAYSCALE_PROB}")
    print(f"PREVIEW_RANDOM_SAMPLES   : {PREVIEW_RANDOM_TRAIN_SAMPLES}")
    print(f"BATCH_SIZE_CANDIDATES    : {BATCH_SIZE_CANDIDATES}")
    print(f"ALLOW_EMPTY_VALTEST      : {ALLOW_EMPTY_VALTEST_CLASSES}")

print_config_block()

In [ ]:
# Cell 2 - Kiem tra environment, venv va NVIDIA GPU
import os
import shutil
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path(PROJECT_DIR).expanduser()
if not PROJECT_DIR.exists():
    raise FileNotFoundError(
        f"PROJECT_DIR khong ton tai: {PROJECT_DIR}. Dat bien moi truong THUOC_PROJECT_DIR dung duong dan du an."
    )

os.chdir(PROJECT_DIR)
print("Working dir:", os.getcwd())
print("Python executable:", sys.executable)

required_files = ["run_all.py", "train_cli.py", "requirements.txt"]
missing = [f for f in required_files if not Path(f).exists()]
if missing:
    raise FileNotFoundError(f"Thieu file bat buoc trong PROJECT_DIR: {missing}")

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    print("\n[nvidia-smi]")
    smi = subprocess.run(
        [nvidia_smi],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
    )
    print(smi.stdout if smi.stdout else smi.stderr)
else:
    print("[WARN] Khong tim thay lenh nvidia-smi trong PATH.")

try:
    import torch  # noqa: F401
except ModuleNotFoundError:
    torch = None
    print("[WARN] Chua cai torch trong venv.")
    print("[ACTION] Hay chay Cell 4 de cai dependencies, sau do chay lai Cell 2.")

if torch is not None:
    print("Torch version:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if DEVICE == "cuda" and not torch.cuda.is_available():
        print("[WARN] Ban chon DEVICE='cuda' nhung CUDA khong san sang. Se fallback sang CPU.")
        RUNTIME_DEVICE = "cpu"
    else:
        RUNTIME_DEVICE = DEVICE

    if torch.cuda.is_available():
        print("CUDA device count:", torch.cuda.device_count())
        print("CUDA current device:", torch.cuda.current_device())
        print("CUDA device name:", torch.cuda.get_device_name(torch.cuda.current_device()))
else:
    # Dat tam theo cau hinh; sau khi cai dependencies, chay lai Cell 2 de validate CUDA.
    RUNTIME_DEVICE = DEVICE

print("RUNTIME_DEVICE:", RUNTIME_DEVICE)

In [ ]:
# Cell 3 - Cap nhat repo (tuy chon)
import subprocess


def run_cmd(cmd: list[str]) -> None:
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True)


if UPDATE_REPO:
    run_cmd(["git", "fetch", "--all"])
    run_cmd(["git", "checkout", REPO_BRANCH])
    run_cmd(["git", "pull", "origin", REPO_BRANCH])
else:
    print("Bo qua git pull (UPDATE_REPO=False)")

print("Repo san sang tai:", PROJECT_DIR)

In [ ]:
# Cell 4 - Cai dependencies trong venv hien tai
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)

import torch
print(f"Python: {sys.version.split()[0]}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# Cell 5 - Chuan bi data dir (uu tien data_aligned tren local)
from pathlib import Path


def _class_dirs(root: Path) -> set[str]:
    if not root.exists():
        return set()
    return {p.name for p in root.iterdir() if p.is_dir()}


def _image_count(class_dir: Path) -> int:
    if not class_dir.exists():
        return 0
    exts = {".jpg", ".jpeg", ".png"}
    return sum(1 for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in exts)


def has_consistent_splits(root: Path) -> bool:
    train_classes = _class_dirs(root / "train")
    val_classes = _class_dirs(root / "val")
    test_classes = _class_dirs(root / "test")
    if not train_classes or not val_classes or not test_classes:
        return False
    return train_classes == val_classes == test_classes


def find_empty_classes(root: Path) -> dict[str, list[str]]:
    result: dict[str, list[str]] = {"train": [], "val": [], "test": []}
    for split in ["train", "val", "test"]:
        split_root = root / split
        if not split_root.exists():
            continue
        for class_name in sorted(_class_dirs(split_root)):
            if _image_count(split_root / class_name) == 0:
                result[split].append(class_name)
    return result


candidate_dirs = [
    PROJECT_DIR / PREFERRED_DATA_DIR,
    PROJECT_DIR / "data_aligned",
    PROJECT_DIR / "data",
]

for extra in EXTRA_DATA_CANDIDATES:
    extra_path = Path(extra).expanduser()
    candidate_dirs.append(extra_path)

data_dir = None
for cand in candidate_dirs:
    if has_consistent_splits(cand):
        data_dir = cand
        break

if data_dir is None:
    raise FileNotFoundError(
        "Khong tim thay data hop le co train/val/test va class folders dong nhat"
    )

train_classes = sorted(_class_dirs(data_dir / "train"))
empty_map = find_empty_classes(data_dir)

split_sizes = {}
for split in ["train", "val", "test"]:
    split_root = data_dir / split
    split_sizes[split] = sum(_image_count(split_root / c) for c in _class_dirs(split_root))

print("Using data dir:", data_dir)
print("Num classes:", len(train_classes))
print("Split sizes:", split_sizes)
print("Empty classes:", {k: len(v) for k, v in empty_map.items()})

if empty_map["train"] or empty_map["val"] or empty_map["test"]:
    print("Empty class names:")
    for split in ["train", "val", "test"]:
        if empty_map[split]:
            print(f" - {split}: {empty_map[split]}")

if empty_map["train"]:
    print("[WARN] Co class rong o train. Model se khong hoc duoc cac class nay.")

if (empty_map["val"] or empty_map["test"]) and not ALLOW_EMPTY_VALTEST_CLASSES:
    raise RuntimeError(
        "Co class rong o val/test. Dat ALLOW_EMPTY_VALTEST_CLASSES=True neu van muon train."
    )

if empty_map["val"] or empty_map["test"]:
    print("[WARN] Phat hien class rong o val/test. Danh gia co the bi lech cho cac class nay.")

## Cell 5b - Data Sanity Preview

Cell nay de kiem tra nhanh anh train truoc khi chay pipeline tren FitLap.
Khong train model o day, chi preview du lieu de phat hien loi som.

In [ ]:
# Cell 5b - Preview random train images (Lab6-style, khong dung model Lab6)
import random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

def _collect_train_images(root: Path, max_per_class: int = 10) -> list[tuple[Path, str]]:
    train_root = root / "train"
    samples = []
    for class_dir in sorted([p for p in train_root.iterdir() if p.is_dir()]):
        imgs = [p for p in class_dir.iterdir() if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}]
        if not imgs:
            continue
        random.shuffle(imgs)
        for p in imgs[:max_per_class]:
            samples.append((p, class_dir.name))
    return samples

if PREVIEW_RANDOM_TRAIN_SAMPLES:
    pool = _collect_train_images(Path(data_dir), max_per_class=12)
    if not pool:
        print("Khong tim thay anh train de preview.")
    else:
        k = min(RANDOM_SAMPLE_COUNT, len(pool))
        picks = random.sample(pool, k)
        cols = 3
        rows = (k + cols - 1) // cols
        plt.figure(figsize=(4.2 * cols, 4.2 * rows))
        for i, (img_path, class_name) in enumerate(picks, start=1):
            plt.subplot(rows, cols, i)
            img = Image.open(img_path).convert("RGB")
            plt.imshow(img)
            plt.title(f"{class_name}\n{img_path.name}", fontsize=9)
            plt.axis("off")
        plt.tight_layout()
        plt.show()
else:
    print("Bo qua preview random train samples.")

In [ ]:
# Cell 6 - Build command theo RUN_MODE
import subprocess
import sys


def pretty_cmd(cmd: list[str]) -> str:
    return " ".join(cmd)


active_device = globals().get("RUNTIME_DEVICE", DEVICE)


def build_run_all_cmd(compare_only: bool = False) -> list[str]:
    cmd = [
        sys.executable,
        "run_all.py",
        "--data-dir", str(data_dir),
        "--device", active_device,
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--eval-batch-size", str(EVAL_BATCH_SIZE),
        "--eval-num-workers", str(NUM_WORKERS),
        "--max-eval-samples", str(MAX_EVAL_SAMPLES),
        "--seed", str(SEED),
        "--output-dir", "models",
        "--augment-profile", AUGMENT_PROFILE,
        "--vector-grayscale-prob", str(float(VECTOR_GRAYSCALE_PROB)),
    ]
    if FAST_TRAIN:
        cmd.append("--fast-train")
    if SKIP_METADATA_ARTIFACTS:
        cmd.append("--skip-metadata-artifacts")
    if compare_only:
        cmd.append("--compare-only")
    return cmd


print("run_all --help")
help_result = subprocess.run(
    [sys.executable, "run_all.py", "--help"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
)
print(help_result.stdout if help_result.stdout else help_result.stderr)

print("\nPreview full command:")
print(pretty_cmd(build_run_all_cmd(compare_only=False)))

if RUN_MODE == "single":
    single_preview = [
        sys.executable,
        "train_cli.py",
        "--mode", "single",
        "--model", SINGLE_MODEL,
        "--data-dir", str(data_dir),
        "--epochs", str(SINGLE_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--device", active_device,
        "--seed", str(SEED),
        "--output-dir", "models",
        "--augment-profile", AUGMENT_PROFILE,
        "--vector-grayscale-prob", str(float(VECTOR_GRAYSCALE_PROB)),
    ]
    print("\nPreview single command:")
    print(pretty_cmd(single_preview))

In [ ]:
# Cell 7 - Chay pipeline
import subprocess
import sys

active_device = globals().get("RUNTIME_DEVICE", DEVICE)

if RUN_MODE == "full":
    cmd = build_run_all_cmd(compare_only=False)
elif RUN_MODE == "compare-only":
    cmd = build_run_all_cmd(compare_only=True)
elif RUN_MODE == "single":
    cmd = [
        sys.executable,
        "train_cli.py",
        "--mode", "single",
        "--model", SINGLE_MODEL,
        "--data-dir", str(data_dir),
        "--epochs", str(SINGLE_EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--num-workers", str(NUM_WORKERS),
        "--device", active_device,
        "--seed", str(SEED),
        "--output-dir", "models",
        "--augment-profile", AUGMENT_PROFILE,
        "--vector-grayscale-prob", str(float(VECTOR_GRAYSCALE_PROB)),
    ]
    if FAST_TRAIN:
        cmd.append("--fast-train")
    if SKIP_METADATA_ARTIFACTS:
        cmd.append("--skip-metadata-artifacts")
else:
    raise ValueError(f"RUN_MODE khong hop le: {RUN_MODE}")

print("Executing:", " ".join(cmd))
subprocess.run(cmd, check=True)

# Sau single train, goi compare-only de cap nhat evaluation summary
if RUN_MODE == "single":
    eval_cmd = build_run_all_cmd(compare_only=True)
    eval_cmd.extend(["--model", SINGLE_MODEL])
    print("Executing:", " ".join(eval_cmd))
    subprocess.run(eval_cmd, check=True)

print("Done")

In [ ]:
# Cell 8 - Kiem tra nhanh output files
import pandas as pd
from pathlib import Path

models_dir = PROJECT_DIR / "models"
summary_csv = models_dir / "results" / "evaluation" / "evaluation_summary.csv"
train_table_csv = models_dir / "results" / "training" / "training_results_table.csv"
report_json = models_dir / "reports" / "latest" / "evaluation_summary.json"

print("models_dir:", models_dir)
print("results/evaluation/evaluation_summary.csv:", summary_csv.exists())
print("results/training/training_results_table.csv:", train_table_csv.exists())
print("reports/latest/evaluation_summary.json:", report_json.exists())

if summary_csv.exists():
    print("\n=== evaluation_summary.csv ===")
    display(pd.read_csv(summary_csv))

if train_table_csv.exists():
    print("\n=== training_results_table.csv ===")
    display(pd.read_csv(train_table_csv))

ckpts = sorted((models_dir / "AI").glob("**/*_epillid_best.pt"))
print("\nCheckpoints:")
for p in ckpts:
    print("-", p.relative_to(models_dir))
print("Total:", len(ckpts))

In [ ]:
# Cell 9 - Ve chart so sanh tu evaluation_summary.csv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

if summary_csv.exists():
    df = pd.read_csv(summary_csv)
    if {"model", "accuracy", "macro_f1"}.issubset(df.columns):
        main_models = ["resnet50", "efficientnet_b0", "vit_b_16"]
        plot_df = df[df["model"].isin(main_models)].copy()

        if not plot_df.empty:
            models = plot_df["model"].tolist()
            acc = plot_df["accuracy"].astype(float).tolist()
            f1 = plot_df["macro_f1"].astype(float).tolist()

            x = np.arange(len(models))
            width = 0.35

            fig, ax = plt.subplots(figsize=(10, 5))
            bars1 = ax.bar(x - width / 2, acc, width, label="Accuracy", color="#1f77b4")
            bars2 = ax.bar(x + width / 2, f1, width, label="Macro F1", color="#d62728")

            ax.set_title("THUOC Model Comparison")
            ax.set_ylabel("Score")
            ax.set_xticks(x)
            ax.set_xticklabels(models)
            ax.set_ylim(0, 1.0)
            ax.grid(axis="y", linestyle="--", alpha=0.3)
            ax.legend()

            for bars in [bars1, bars2]:
                for b in bars:
                    h = b.get_height()
                    ax.annotate(f"{h:.3f}", (b.get_x() + b.get_width() / 2, h), ha="center", va="bottom", fontsize=9)

            plt.tight_layout()
            plt.show()
        else:
            print("Khong co du lieu 3 model chinh trong evaluation_summary.csv")
    else:
        print("evaluation_summary.csv thieu cot model/accuracy/macro_f1")
else:
    print("Chua co evaluation_summary.csv. Hay chay Cell 7 truoc.")

In [ ]:
# Cell 10 - Dong goi artifacts ra file zip (tuy chon)
import shutil
from pathlib import Path

archive_base = PROJECT_DIR / "thuoc_models_artifacts_fitlap"
archive_path = shutil.make_archive(
    str(archive_base),
    "zip",
    root_dir=str(PROJECT_DIR / "models"),
)
print("Created:", archive_path)
print("Ban co the tai file zip tai:", archive_path)

In [ ]:
# Cell 11 - Lenh mau de chay tay khi can (khong bat buoc)
print("Manual commands (FitLap):")
print("1) Kich hoat venv")
print("   source .venv/bin/activate")
print("2) Kiem tra GPU")
print("   nvidia-smi")
print("3) Full pipeline")
print("   python run_all.py --data-dir data_aligned --device cuda --batch-size 16")
print("4) Compare only (checkpoints da co)")
print("   python run_all.py --compare-only --data-dir data_aligned --device cuda")
print("5) Single model")
print("   python train_cli.py --mode single --model resnet50 --epochs 12 --data-dir data_aligned")
print("6) Fast train for quick debug")
print("   python run_all.py --data-dir data_aligned --device cpu --fast-train --max-eval-samples 400")
print("7) Batch size test (Lab6-inspired workflow, khong su dung model Lab6)")
print("   python train_cli.py --mode single --model resnet50 --epochs 4 --batch-size 16 --data-dir data_aligned")
print("   python train_cli.py --mode single --model resnet50 --epochs 4 --batch-size 24 --data-dir data_aligned")
print("   python train_cli.py --mode single --model resnet50 --epochs 4 --batch-size 32 --data-dir data_aligned")
print("8) Artifact paths")
print("   models/results/evaluation/evaluation_summary.csv")
print("   models/results/training/training_results_table.csv")
print("   models/AI/<model_alias>/*_epillid_best.pt")